In [ ]:
DATA = 'data.json'
URL = 'https://collectionapi.metmuseum.org/public/collection/v1/'

In [ ]:
%pip install requests google-genai

In [ ]:
import requests
import json

def get_id_data():
    response = requests.get(f'{URL}/objects')
    dados = response.json()
    return dados['objectIDs']

def get_images(id):
    images = requests.get(f'{URL}/objects/{id}')
    link = images.json().get('primaryImage')
    return link


In [ ]:
import random

ids = get_id_data()

image_valid = None
while not image_valid:
    id_random = random.choice(ids)
    image_valid = get_images(id_random)

print(image_valid)


In [ ]:
from google import genai
from google.genai import types
import config

client = genai.Client(api_key=config.GEMINI_API_KEY)

download = requests.get(image_valid)

mime_type = download.headers.get('Content-Type', 'image/jpeg').split(';')[0]

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Part.from_bytes(
            data=download.content,
            mime_type=mime_type
        ),
        "O que você vê nessa imagem?"
    ]
)

print(response.text)

In [ ]:
import os

new_data = {
    'id':id_random,
    'link': image_valid,
    'description': response.text
}

if os.path.exists(DATA):
    with open(DATA, 'r', encoding='utf-8') as f:
        try:
            existing_data = json.load(f)
        except json.JSONDecodeError:
            existing_data = []
else:
    existing_data = []

existing_data.append(new_data)

with open(DATA, 'w', encoding='utf-8') as f:
    json.dump(existing_data, f, indent=4, ensure_ascii=False)

print(f'ID: {id_random} e Link: {image_valid}')